# 03 - Modeling
Pemilihan k (elbow + silhouette + Davies-Bouldin) dan pelatihan K-Means.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rfm_kmeans import config
cfg = config.load_config()
SEED = cfg["seed"]
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)
from rfm_kmeans import modeling, visualization
OUT = config.resolve(cfg, "processed_dir")
TAB = config.resolve(cfg, "tables_dir"); TAB.mkdir(parents=True, exist_ok=True)
FIG = config.resolve(cfg, "figures_dir")
rfm = pd.read_csv(OUT / "rfm.csv")
X = pd.read_csv(OUT / "rfm_scaled.csv", index_col=0)

## Pemilihan k

In [2]:
m = cfg["modeling"]
scan = modeling.scan_k(X, m["k_min"], m["k_max"], SEED, m["n_init"])
scan.to_csv(TAB / "k_selection.csv", index=False)
visualization.plot_k_selection(scan, FIG / "k_selection.png")
plt.show()
scan

/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_96776/1404452808.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,k,inertia,silhouette,davies_bouldin
0,2,876.59,0.43,0.89
1,3,673.88,0.32,1.09
2,4,544.83,0.32,1.02
3,5,451.72,0.32,1.01
4,6,394.04,0.31,1.02
5,7,354.36,0.31,1.00
6,8,321.47,0.31,0.99
7,9,298.13,0.30,1.01
8,10,275.97,0.30,1.01


In [3]:
k = modeling.choose_k(scan, m["k_candidates"])
print("k terpilih (silhouette tertinggi pada 3-6):", k)
# Bila hasil dipertimbangkan ulang secara bisnis, isi manual: k = 4

k terpilih (silhouette tertinggi pada 3-6): 5


## Latih model akhir

In [4]:
model = modeling.fit_kmeans(X, k, SEED, m["n_init"], m["max_iter"])
rfm["Cluster"] = model.labels_
rfm["Cluster"].value_counts().sort_index()

Cluster
0     62
1    124
2    128
3    125
4    154
Name: count, dtype: int64

In [5]:
visualization.plot_clusters_pca(X, model.labels_, FIG / "clusters_pca.png")
plt.show()

/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_96776/2260995693.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
rfm.to_csv(OUT / "rfm_clustered.csv", index=False)